In [32]:
import os

In [33]:
%pwd

'c:\\Users\\chait\\ui\\FirstProj_MLOPS'

In [34]:
#os.chdir("../")

In [35]:
import pandas as pd;
data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv");

In [36]:
print(data.head());

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2      9.8        5 

In [37]:
data.info();

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [38]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [39]:
data.shape

(1599, 12)

In [40]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir:Path
    status_file: str
    unzip_data_dir: Path
    all_schema:dict

In [41]:
from src.datasci.constants import *
from src.datasci.utils.common import read_yaml, create_directories

In [42]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath);
        self.params = read_yaml(params_filepath);
        self.schema = read_yaml(schema_filepath);

        create_directories([self.config.artifacts_root])

    def getDataValidationConfig(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS
        create_directories([config.root_dir]);
        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            status_file=config.status_file,
            unzip_data_dir=config.unzip_data_dir,
            all_schema = schema,
        )
        return data_validation_config;
        

In [43]:
import os
from src.datasci import logger

In [44]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self)->bool:
        try:
            validation_status = None
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns);
            all_schema = self.config.all_schema.keys();

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.status_file, 'w') as f:
                        f.write(f"Validation Status: {validation_status}");
                else:
                    validation_status = True;
                    with open(self.config.status_file, 'w') as f:
                        f.write(f"Validation status: {validation_status}");
            return validation_status;
        except Exception as e:
            raise e


In [46]:
try:
    config = ConfigurationManager();
    data_validation_config = config.getDataValidationConfig();
    data_validation = DataValidation(config=data_validation_config);
  #  print(data_validation.config);
    data_validation.validate_all_columns();
except Exception as e:
    raise e

[2026-08-30 13:09:07,954: INFO: common: yaml file: config\config.yaml loaded successful]
[2026-08-30 13:09:07,958: INFO: common: yaml file: params.yaml loaded successful]
[2026-08-30 13:09:07,962: INFO: common: yaml file: schema.yaml loaded successful]
[2026-08-30 13:09:07,964: INFO: common: created directory at: artifacts]
[2026-08-30 13:09:07,966: INFO: common: created directory at: artifacts/data_validation]
